In [ ]:
# import os
# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [2]:
from datetime import UTC, datetime

import biogeme.biogeme as bio
import biogeme.biogeme_logging as blog
import biogeme.database as db
import numpy as np
import pandas as pd
from biogeme import models
from biogeme.expressions import Beta, Variable, log


/home/yianzhang/work/research/migration/migration/.pixi/envs/default/lib/python3.12/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
/home/yianzhang/work/research/migration/migration/.pixi/envs/default/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
unixtime = int(datetime.now(UTC).timestamp())

In [4]:
year = 2018
num_alternatives = 50

### Read data, only keep used columns

In [ ]:
# same columns= restriction technique as modeling_xlogit.ipynb: read only the columns V[0]/V[i]
# actually reference, instead of the full ~3,800-column file, since most of it is unused census
# fields. ALT_VARYING_SUFFIXES covers the ALT{i}_<suffix> destination columns V[i] uses;
# INDIV_COLS covers everything else, plus CHOSEN/STAY/ALT{i}_PUMA needed to build ALT_CHOICE below.
ALT_VARYING_SUFFIXES = [
    "ALT_COMMUTE_PCT",
    "CBSA",
    "COLLEGE_PCT",
    "DIST",
    "ENT_JOBS_PCT",
    "FOREIGN_BORN_PCT",
    "HH_MED_INC",
    "HH_WITH_CHILD_PCT",
    "HOUSE_VACANCY_PCT",
    "MED_HOUSE_VALUE_OVER_MED_HH_INC",
    "MED_RENT_PCT_HH_INC",
    "MED_TRAVEL_TIME",
    "MIL_PCT",
    "OWN_AGE_PCT",
    "OWN_GROUP_PCT",
    "OWN_RACE_PCT",
    "STATE",
    "TOT_POP",
    "TYPE",
    "UNEMP_RATE",
]

INDIV_COLS = [
    "CHOSEN",
    "STAY",
    "AAPI",
    "AGE_18_22",
    "AGE_18_34",
    "AGE_23_29",
    "AGE_30_39",
    "AGE_35_64",
    "AGE_40_49",
    "AGE_50_64",
    "AGE_OVER_65",
    "BLACK",
    "CHILD",
    "CHILD_6_TO_17",
    "CHILD_UNDER_6",
    "EDU_BACHELORS",
    "EDU_HIGH",
    "EDU_NOHIGH",
    "FOREIGN",
    "House vacancy proportion.ORIG",
    "INDIAN",
    "IN_COLLEGE",
    "IN_MILITARY",
    "LATINO",
    "MARRIED_MORE_THAN_YEAR",
    "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG",
    "Median gross rent as a percentage of household income.ORIG",
    "Median house value over median household income.ORIG",
    "Median travel time.ORIG",
    "NAICS_AGR_EXT",
    "NAICS_GOODS_TRADE",
    "NAICS_GOVT",
    "NAICS_GROUP_PCT_AGR_EXT.ORIG",
    "NAICS_GROUP_PCT_GOODS_TRADE.ORIG",
    "NAICS_GROUP_PCT_GOVT.ORIG",
    "NAICS_GROUP_PCT_HIGH_ED.ORIG",
    "NAICS_GROUP_PCT_LICENSE.ORIG",
    "NAICS_HIGH_ED",
    "NAICS_LICENSE",
    "NAME_NUM.ORIG",
    "POBP",
    "Proportion alternative commute.ORIG",
    "Proportion foreign born.ORIG",
    "Proportion of households with children.ORIG",
    "Proportion of people 18-34.ORIG",
    "Proportion of people 35-64.ORIG",
    "Proportion of people 65+.ORIG",
    "Proportion of people AAPI.ORIG",
    "Proportion of people Black.ORIG",
    "Proportion of people Indian.ORIG",
    "Proportion of people Latino.ORIG",
    "Proportion of people other race.ORIG",
    "Proportion of people in college.ORIG",
    "Proportion of people in military.ORIG",
    "RECENTLY_MARRIED",
    "RECENTLY_WIDOWED_OR_DIVORCED",
    "SINGLE_PARENT",
    "ST",
    "TYPE_NUM.ORIG",
    "Unemployment rate.ORIG",
    "WORK1_MAR",
    "WORK2_MAR",
    "Total Population.Total Population.SE_A00001_001.ORIG",
    "Proportion of entertainment jobs.ORIG",
]

needed_cols = (
    INDIV_COLS
    + [f"ALT{i}_PUMA" for i in range(1, num_alternatives + 1)]
    + [
        f"ALT{i}_{suf}"
        for i in range(1, num_alternatives + 1)
        for suf in ALT_VARYING_SUFFIXES
    ]
)
needed_cols = list(dict.fromkeys(needed_cols))
print(f"reading {len(needed_cols)} columns")

df = pd.read_parquet(f"../data/us_estdata_{year}.parquet", columns=needed_cols)
df


reading 1264 columns


,CHOSEN,STAY,AAPI,AGE_18_22,AGE_18_34,AGE_23_29,AGE_30_39,AGE_35_64,AGE_40_49,AGE_50_64,...,ALT50_MIL_PCT,ALT50_OWN_AGE_PCT,ALT50_OWN_GROUP_PCT,ALT50_OWN_RACE_PCT,ALT50_POVERTY_PCT,ALT50_STATE,ALT50_TOT_POP,ALT50_TYPE,ALT50_UNEMP_RATE,ALT50_YR_SINCE_MED_STRUCTURE
0,1701104,0,0,0,0,0,0,1,0,1,...,0.000000,0.412433,0.295632,0.884180,0.190228,47,128091.0,1,0.037901,30.0
1,2400505,1,1,0,0,0,0,0,0,0,...,0.000000,0.165795,0.000000,0.119423,0.129263,25,131560.0,0,0.044980,74.0
2,0607310,1,0,0,0,0,1,1,0,0,...,0.000000,0.357529,0.444816,0.585089,0.394878,12,125092.0,1,0.046581,28.0
3,3604110,1,0,0,0,0,0,1,0,1,...,0.000000,0.334424,0.237939,0.103205,0.626866,48,130532.0,0,0.098991,54.0
4,5151167,0,0,0,0,0,0,1,1,0,...,0.000931,0.382783,0.171376,0.308453,0.471484,05,100608.0,2,0.037477,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253085,2401400,1,0,0,0,0,0,1,1,0,...,0.000000,0.308822,0.038703,0.705051,0.466067,21,113185.0,1,0.069069,53.0
253086,0800900,1,0,0,1,1,0,0,0,0,...,0.000000,0.165703,0.349125,0.899581,0.133759,42,115647.0,0,0.033786,50.0
253087,0800824,1,0,0,0,0,0,1,0,1,...,0.000846,0.439306,0.073406,0.548122,0.231590,06,117722.0,0,0.039054,59.0
253088,4805915,1,0,0,0,0,0,1,0,1,...,0.000854,0.394203,0.236879,0.772463,0.241918,24,171904.0,2,0.038792,38.0


### Clean data, make everything numeric

In [6]:
cat_cols = df.dtypes[df.dtypes == "category"].keys()
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype(int))

obj_cols = list(df.dtypes[df.dtypes == "object"].keys())
df[obj_cols] = df[obj_cols].apply(lambda x: x.astype(int))

In [7]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df["NAME_NUM.ORIG"].min(), df["ALT1_CBSA"].min()

(np.int64(-2), np.int16(-1))

In [8]:
# clean up the database (Biogeme Database can only have numerical values)
df_train = df.select_dtypes(["number"])
assert df_train.isna().sum().sum() == 0
del df

### Look at collinearity in requested columns

In [ ]:
# stack all alternatives into long format
frames = []
for i in range(1, num_alternatives + 1):
    cols = {
        f"ALT{i}_{v}": v
        for v in ALT_VARYING_SUFFIXES
        if f"ALT{i}_{v}" in df_train.columns
    }
    frames.append(df_train[list(cols)].rename(columns=cols))

long = pd.concat(frames, ignore_index=True)

# add the transformed versions you actually use in the model
long["log_DIST"] = np.log(long["DIST"] + 1)
long["log_TOT_POP"] = np.log(long["TOT_POP"])

corr = long.corr()

In [14]:
c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3])

TOT_POP                          log_TOT_POP                  0.992355
DIST                             log_DIST                     0.881614
HH_MED_INC                       POVERTY_PCT                  0.855815
MED_HOUSE_VALUE_OVER_MED_HH_INC  FOREIGN_BORN_PCT             0.615930
POVERTY_PCT                      UNEMP_RATE                   0.613056
MED_HOUSE_VALUE_OVER_MED_HH_INC  ALT_COMMUTE_PCT              0.597039
MED_TRAVEL_TIME                  FOREIGN_BORN_PCT             0.541512
YR_SINCE_MED_STRUCTURE           ALT_COMMUTE_PCT              0.474177
HH_MED_INC                       UNEMP_RATE                   0.470170
MED_HOUSE_VALUE_OVER_MED_HH_INC  MED_TRAVEL_TIME              0.433662
HH_MED_INC                       HOUSE_VACANCY_PCT            0.431286
MED_TRAVEL_TIME                  ALT_COMMUTE_PCT              0.423422
HOUSE_VACANCY_PCT                HH_WITH_CHILD_PCT            0.407723
MED_OWNER_COST_HH_INC_PCT        FOREIGN_BORN_PCT             0.402411
HH_MED

In [18]:
orig_vars = [v for v in INDIV_COLS if v in df_train.columns]  # skip any missing
corr = df_train[orig_vars].corr()

c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.3].to_string())

Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG  Proportion of people struggling.ORIG                                                                                                            0.850457
Proportion of people 18-34.ORIG                                                                                                               Proportion of people in college.ORIG                                                                                                            0.800195
Proportion foreign born.ORIG                                                                                                                  Median house value over median household income.ORIG                                                                                            0.752637
Proportion of people AAPI.ORIG                                                                                     

### Creating alternatives

In [8]:
# defining the chosen alterantive for each person explicitly
# each person has chosen from alterantives 0-num_alterantives, 0 represents staying and 1-num_alterantives represent moving to the PUMAs each represents
# for movers, they all choose alterantive 1 by construction; all stayers choose alterantive 0
df_train["ALT_CHOICE"] = 0
for i in range(1, num_alternatives + 1):
    var = "ALT" + str(i) + "_PUMA"
    df_train["ALT_CHOICE"] = np.where(
        df_train[var] == df_train["CHOSEN"], i, df_train["ALT_CHOICE"]
    )
df_train["ALT_CHOICE"] = np.where(df_train["STAY"] == 1, 0, df_train["ALT_CHOICE"])

### Setting up biogeme & utilities

In [13]:
# making the Biogeme Database that is used for the model estimation
database = db.Database("us_data", df_train)

In [ ]:
c_stay = Beta("c_stay", 0, None, None, 0)

# age
c_stay_age_18_22 = Beta("c_stay_age_18_22", 0, None, None, 0)
c_stay_age_23_29 = Beta("c_stay_age_23_29", 0, None, None, 0)
c_stay_age_30_39 = Beta("c_stay_age_30_39", 0, None, None, 0)
c_stay_age_40_49 = Beta("c_stay_age_40_49", 0, None, None, 0)
c_stay_age_50_64 = Beta("c_stay_age_50_64", 0, None, None, 0)
# c_stay_age_65 = Beta("c_stay_age_65", 0, None, None, 1)

# personal lives
c_stay_child_under_6 = Beta("c_stay_child_under_6", 0, None, None, 0)
c_stay_child_6_to_17 = Beta("c_stay_child_6_to_17", 0, None, None, 0)

c_stay_married_more_than_year = Beta("c_stay_married_more_than_year", 0, None, None, 0)
c_stay_married_less_than_year = Beta("c_stay_married_less_than_year", 0, None, None, 0)
c_stay_recently_divorced_or_widowed = Beta(
    "c_stay_recently_divorced_or_widowed", 0, None, None, 0
)
c_stay_2work_mar = Beta("c_stay_2work_mar", 0, None, None, 0)
# reference
# c_stay_1work_mar = Beta("c_stay_1work_mar", 0, None, None, 0)
c_stay_single_parent = Beta("c_stay_single_parent", 0, None, None, 0)


# education
c_stay_edu_college = Beta("c_stay_edu_college", 0, None, None, 0)
c_stay_edu_high = Beta("c_stay_edu_high", 0, None, None, 0)
# reference
# c_stay_edu_nohigh = Beta("c_stay_edu_nohigh", 0, None, None, 0)

c_stay_in_college = Beta("c_stay_in_college", 0, None, None, 0)

# race & foreign status
c_stay_foreign = Beta("c_stay_foreign", 0, None, None, 0)

# NOTE: this assumes that NAICS code stays constant between the origin and destination
c_stay_mil = Beta("c_stay_mil", 0, None, None, 0)
c_stay_naics_govt = Beta("c_stay_naics_govt", 0, None, None, 0)
c_stay_naics_goods_trade = Beta("c_stay_naics_goods_trade", 0, None, None, 0)
c_stay_naics_license = Beta("c_stay_naics_license", 0, None, None, 0)
c_stay_naics_high_ed = Beta("c_stay_naics_high_ed", 0, None, None, 0)
c_stay_naics_agr_ext = Beta("c_stay_naics_agr_ext", 0, None, None, 0)
# low svc proportion & unemp/not in lf reference; shouldn't be very relevant

In [ ]:
# race & age similarity
c_proportion_same_age_18_34 = Beta("c_proportion_same_age_18_34", 0, None, None, 0)
c_proportion_same_age_35_64 = Beta("c_proportion_same_age_35_64", 0, None, None, 0)
c_proportion_same_age_65_plus = Beta("c_proportion_same_age_65_plus", 0, None, None, 0)
c_proportion_same_race_black = Beta("c_proportion_same_race_black", 0, None, None, 0)
c_proportion_same_race_aapi = Beta("c_proportion_same_race_aapi", 0, None, None, 0)
c_proportion_same_race_indian = Beta("c_proportion_same_race_indian", 0, None, None, 0)
c_proportion_also_latino = Beta("c_proportion_also_latino", 0, None, None, 0)
c_proportion_same_race_white = Beta("c_proportion_same_race_white", 0, None, None, 0)
c_proportion_same_race_other = Beta("c_proportion_same_race_other", 0, None, None, 0)

# coefficients for certain groups
c_proportion_hh_with_children_if_have_children = Beta(
    "c_proportion_hh_with_children_if_have_children", 0, None, None, 0
)
c_proportion_college_if_in_college = Beta(
    "c_proportion_college_if_in_college", 0, None, None, 0
)
c_proportion_foreign_if_foreign = Beta(
    "c_proportion_foreign_if_foreign", 0, None, None, 0
)

# economic indicators
c_median_hh_income_in_tens_of_thousands = Beta(
    "c_median_hh_income_in_tens_of_thousands", 0, None, None, 0
)
c_median_house_value_over_median_income = Beta(
    "c_median_house_value_over_median_income", 0, None, None, 0
)
c_median_gross_rent_percentage_hh_inc = Beta(
    "c_median_gross_rent_percentage_hh_inc", 0, None, None, 0
)
c_unemp_rate = Beta("c_unemp_rate", 0, None, None, 0)
c_vacancy_rate = Beta("c_vacancy_rate", 0, None, None, 0)

# quality of life
c_median_travel_time = Beta("c_median_travel_time", 0, None, None, 0)
c_proportion_alt_commute = Beta("c_proportion_alt_commute", 0, None, None, 0)

c_proportion_ent = Beta("c_proportion_ent", 0, None, None, 0)
# these two are relative ot the effect for over 65 (captured by ent)
c_proportion_ent_18_34 = Beta("c_proportion_ent_18_34", 0, None, None, 0)
c_proportion_ent_35_64 = Beta("c_proportion_ent_35_64", 0, None, None, 0)

# jobs
c_proportion_also_mil = Beta("c_proportion_also_mil", 0, None, None, 0)
c_proportion_same_naics_govt = Beta("c_proportion_same_naics_govt", 0, None, None, 0)
c_proportion_same_naics_goods_trade = Beta(
    "c_proportion_same_naics_goods_trade", 0, None, None, 0
)
c_proportion_same_naics_license = Beta(
    "c_proportion_same_naics_license", 0, None, None, 0
)
c_proportion_same_naics_high_ed = Beta(
    "c_proportion_same_naics_high_ed", 0, None, None, 0
)
c_proportion_same_naics_agr_ext = Beta(
    "c_proportion_same_naics_agr_ext", 0, None, None, 0
)

# dropped variables
# this is extremely collinear to household income
# c_proportion_struggling = Beta("c_proportion_struggling", 0, None, None, 0)
# c_destchoice_yrs_since_median_structure_built = Beta(
#     "c_yrs_since_median_structure_built", 0, None, None, 0
# )
# c_median_owner_costs = Beta("c_median_owner_costs", 0, None, None, 0)
# c_median_travel_time = Beta("c_median_travel_time", 0, None, None, 0)

In [15]:
# dictionary of utilities; number -> utility function
V = {}

In [ ]:
# defining the staying utility function
V[0] = (
    # constant
    c_stay
    # size term
    + log(Variable("Total Population.Total Population.SE_A00001_001.ORIG"))
    # age categories
    + c_stay_age_18_22 * Variable("AGE_18_22")
    + c_stay_age_23_29 * Variable("AGE_23_29")
    + c_stay_age_30_39 * Variable("AGE_30_39")
    + c_stay_age_40_49 * Variable("AGE_40_49")
    + c_stay_age_50_64 * Variable("AGE_50_64")
    # age similarity to origin
    + c_proportion_same_age_18_34
    * Variable("Proportion of people 18-34.ORIG")
    * Variable("AGE_18_34")
    + c_proportion_same_age_35_64
    * Variable("Proportion of people 35-64.ORIG")
    * Variable("AGE_35_64")
    + c_proportion_same_age_65_plus
    * Variable("Proportion of people 65+.ORIG")
    * Variable("AGE_OVER_65")
    # personal life situations
    + c_stay_child_under_6 * Variable("CHILD_UNDER_6")
    + c_stay_child_6_to_17 * Variable("CHILD_6_TO_17")
    + c_proportion_hh_with_children_if_have_children
    * Variable("Proportion of households with children.ORIG")
    * Variable("CHILD")
    + c_stay_married_more_than_year * Variable("MARRIED_MORE_THAN_YEAR")
    + c_stay_married_less_than_year * Variable("RECENTLY_MARRIED")
    + c_stay_recently_divorced_or_widowed * Variable("RECENTLY_WIDOWED_OR_DIVORCED")
    + c_stay_2work_mar * Variable("WORK2_MAR")
    + c_stay_single_parent * Variable("SINGLE_PARENT")
    # education
    + c_stay_edu_college * Variable("EDU_BACHELORS")
    + c_stay_edu_high * Variable("EDU_HIGH")
    + c_stay_in_college * Variable("IN_COLLEGE")
    + c_proportion_college_if_in_college
    * Variable("Proportion of people in college.ORIG")
    * Variable("IN_COLLEGE")
    # race & foreign status
    + c_stay_foreign * Variable("FOREIGN")
    + c_proportion_same_race_black
    * Variable("Proportion of people Black.ORIG")
    * Variable("BLACK")
    + c_proportion_same_race_aapi
    * Variable("Proportion of people AAPI.ORIG")
    * Variable("AAPI")
    + c_proportion_same_race_indian
    * Variable("Proportion of people Indian.ORIG")
    * Variable("INDIAN")
    + c_proportion_also_latino
    * Variable("Proportion of people Latino.ORIG")
    * Variable("LATINO")
    + c_proportion_same_race_white
    * Variable("Proportion of people White.ORIG")
    * Variable("WHITE")
    + c_proportion_same_race_other * Variable("Proportion of ")
    + c_proportion_foreign_if_foreign
    * Variable("Proportion foreign born.ORIG")
    * Variable("FOREIGN")
    # economic & housing indicators
    + c_median_hh_income_in_tens_of_thousands
    * Variable(
        "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
    )
    / 10_000
    + c_median_house_value_over_median_income
    * Variable("Median house value over median household income.ORIG")
    + c_median_gross_rent_percentage_hh_inc
    * Variable("Median gross rent as a percentage of household income.ORIG")
    + c_unemp_rate * Variable("Unemployment rate.ORIG")
    + c_vacancy_rate * Variable("House vacancy proportion.ORIG")
    # quality of life
    + c_median_travel_time * Variable("Median travel time.ORIG")
    + c_proportion_alt_commute * Variable("Proportion alternative commute.ORIG")
    + c_proportion_ent * Variable("Proportion of entertainment jobs.ORIG")
    + c_proportion_ent_18_34
    * Variable("AGE_18_34")
    * Variable("Proportion of entertainment jobs.ORIG")
    + c_proportion_ent_35_64
    * Variable("AGE_35_64")
    * Variable("Proportion of entertainment jobs.ORIG")
    # NAICS industry group
    + c_stay_mil * Variable("IN_MILITARY")
    + c_stay_naics_govt * Variable("NAICS_GOVT")
    + c_stay_naics_goods_trade * Variable("NAICS_GOODS_TRADE")
    + c_stay_naics_license * Variable("NAICS_LICENSE")
    + c_stay_naics_high_ed * Variable("NAICS_HIGH_ED")
    + c_stay_naics_agr_ext * Variable("NAICS_AGR_EXT")
    + c_proportion_also_mil
    * Variable("Proportion of people in military.ORIG")
    * Variable("IN_MILITARY")
    + c_proportion_same_naics_govt
    * Variable("NAICS_GROUP_PCT_GOVT.ORIG")
    * Variable("NAICS_GOVT")
    + c_proportion_same_naics_goods_trade
    * Variable("NAICS_GROUP_PCT_GOODS_TRADE.ORIG")
    * Variable("NAICS_GOODS_TRADE")
    + c_proportion_same_naics_license
    * Variable("NAICS_GROUP_PCT_LICENSE.ORIG")
    * Variable("NAICS_LICENSE")
    + c_proportion_same_naics_high_ed
    * Variable("NAICS_GROUP_PCT_HIGH_ED.ORIG")
    * Variable("NAICS_HIGH_ED")
    + c_proportion_same_naics_agr_ext
    * Variable("NAICS_GROUP_PCT_AGR_EXT.ORIG")
    * Variable("NAICS_AGR_EXT")
)

In [ ]:
# Destination Choice Parameters to be estimated

# geography
# c_destchoice_dist = Beta("c_destchoice_dist", 0, None, None, 0)
c_destchoice_logdist = Beta("c_destchoice_logdist", 0, None, None, 0)

c_destchoice_samestate = Beta("c_destchoice_samestate", 0, None, None, 0)
c_destchoice_birthstate = Beta("c_destchoice_birthstate", 0, None, None, 0)
c_destchoice_samecbsa = Beta("c_destchoice_samecbsa", 0, None, None, 0)

# differences between density
c_destchoice_T34_T34 = Beta("c_destchoice_T34_T34", 0, None, None, 0)
c_destchoice_T34_metro = Beta("c_destchoice_T34_metro", 0, None, None, 0)
c_destchoice_T34_nonmetro = Beta("c_destchoice_T34_nonmetro", 0, None, None, 0)
c_destchoice_metro_T34 = Beta("c_destchoice_metro_T34", 0, None, None, 0)
c_destchoice_metro_metro = Beta("c_destchoice_metro_metro", 0, None, None, 0)
c_destchoice_metro_nonmetro = Beta("c_destchoice_metro_nonmetro", 0, None, None, 0)
c_destchoice_nonmetro_T34 = Beta("c_destchoice_nonmetro_T34", 0, None, None, 0)
c_destchoice_nonmetro_metro = Beta("c_destchoice_nonmetro_metro", 0, None, None, 0)
# c_destchoice_nonmetro_nonmetro = Beta(
#     "c_destchoice_nonmetro_nonmetro", 0, None, None, 0
# )

In [ ]:
# defining the utility functions for each of the moving PUMA alternatives
for i in range(1, num_alternatives + 1):
    alt = f"ALT{i}_"

    same_state = Variable("ST") == Variable(f"{alt}STATE")
    # NAME_NUM.ORIG and ALT{i}_CBSA are factorized against the same CBSA-name codebook
    # (see create_estdata.ipynb), so they're directly comparable
    same_cbsa = Variable("NAME_NUM.ORIG") == Variable(f"{alt}CBSA")
    same_type_t34 = Variable("TYPE_NUM.ORIG") == 0
    same_type_metro = Variable("TYPE_NUM.ORIG") == 1
    same_type_nonmetro = Variable("TYPE_NUM.ORIG") == 2
    alt_type_t34 = Variable(f"{alt}TYPE") == 0
    alt_type_metro = Variable(f"{alt}TYPE") == 1
    alt_type_nonmetro = Variable(f"{alt}TYPE") == 2

    V[i] = (
        # size term
        log(Variable(f"{alt}TOT_POP"))
        # geography
        + c_destchoice_logdist * log(Variable(f"{alt}DIST") + 1)
        + c_destchoice_samecbsa * same_cbsa
        + c_destchoice_samestate * same_state
        + c_destchoice_birthstate * (Variable("POBP") == Variable(f"{alt}STATE"))
        # economic
        + c_median_hh_income_in_tens_of_thousands
        * Variable(f"{alt}HH_MED_INC")
        / 10_000
        + c_median_house_value_over_median_income
        * Variable(f"{alt}MED_HOUSE_VALUE_OVER_MED_HH_INC")
        + c_median_gross_rent_percentage_hh_inc * Variable(f"{alt}MED_RENT_PCT_HH_INC")
        + c_unemp_rate * Variable(f"{alt}UNEMP_RATE")
        + c_vacancy_rate * Variable(f"{alt}HOUSE_VACANCY_PCT")
        # similarity to migrant
        + c_proportion_college_if_in_college
        * Variable("IN_COLLEGE")
        * Variable(f"{alt}COLLEGE_PCT")
        + c_proportion_foreign_if_foreign
        * Variable("FOREIGN")
        * Variable(f"{alt}FOREIGN_BORN_PCT")
        + c_proportion_hh_with_children_if_have_children
        * Variable(f"{alt}HH_WITH_CHILD_PCT")
        * Variable("CHILD")
        # age similarity
        + c_proportion_same_age_18_34
        * Variable("AGE_18_34")
        * Variable(f"{alt}OWN_AGE_PCT")
        + c_proportion_same_age_35_64
        * Variable("AGE_35_64")
        * Variable(f"{alt}OWN_AGE_PCT")
        + c_proportion_same_age_65_plus
        * Variable("AGE_OVER_65")
        * Variable(f"{alt}OWN_AGE_PCT")
        # race similarity
        + c_proportion_same_race_black
        * Variable("BLACK")
        * Variable(f"{alt}OWN_RACE_ETH_PCT")
        + c_proportion_same_race_aapi
        * Variable("AAPI")
        * Variable(f"{alt}OWN_RACE_ETH_PCT")
        + c_proportion_same_race_indian
        * Variable("INDIAN")
        * Variable(f"{alt}OWN_RACE_ETH_PCT")
        + c_proportion_also_latino
        * Variable("LATINO")
        * Variable(f"{alt}OWN_RACE_ETH_PCT")
        + c_proportion_same_race_white
        * Variable("Proportion of people White.ORIG")
        * Variable("WHITE")
        # quality of life
        + c_proportion_ent * Variable(f"{alt}ENT_JOBS_PCT")
        + c_proportion_ent_18_34
        * Variable("AGE_18_34")
        * Variable(f"{alt}ENT_JOBS_PCT")
        + c_proportion_ent_35_64
        * Variable("AGE_35_64")
        * Variable(f"{alt}ENT_JOBS_PCT")
        + c_median_travel_time * Variable(f"{alt}MED_TRAVEL_TIME")
        + c_proportion_alt_commute * Variable(f"{alt}ALT_COMMUTE_PCT")
        # origin area type x destination area type
        + c_destchoice_T34_T34 * same_type_t34 * alt_type_t34
        + c_destchoice_T34_metro * same_type_t34 * alt_type_metro
        + c_destchoice_T34_nonmetro * same_type_t34 * alt_type_nonmetro
        + c_destchoice_metro_T34 * same_type_metro * alt_type_t34
        + c_destchoice_metro_metro * same_type_metro * alt_type_metro
        + c_destchoice_metro_nonmetro * same_type_metro * alt_type_nonmetro
        + c_destchoice_nonmetro_T34 * same_type_nonmetro * alt_type_t34
        + c_destchoice_nonmetro_metro * same_type_nonmetro * alt_type_metro
        # NAICS
        + c_proportion_also_mil * Variable("IN_MILITARY") * Variable(f"{alt}MIL_PCT")
        + c_proportion_same_naics_govt
        * Variable("NAICS_GOVT")
        * Variable(f"{alt}OWN_NAICS_GROUP_PCT")
        + c_proportion_same_naics_goods_trade
        * Variable("NAICS_GOODS_TRADE")
        * Variable(f"{alt}OWN_GROUPOWN_NAICS_GROUP_PCT_PCT")
        + c_proportion_same_naics_license
        * Variable("NAICS_LICENSE")
        * Variable(f"{alt}OWN_NAICS_GROUP_PCT")
        + c_proportion_same_naics_high_ed
        * Variable("NAICS_HIGH_ED")
        * Variable(f"{alt}OWN_NAICS_GROUP_PCT")
        + c_proportion_same_naics_agr_ext
        * Variable("NAICS_AGR_EXT")
        * Variable(f"{alt}OWN_NAICS_GROUP_PCT")
    )


In [19]:
# all alternatives are available
av = {}
for i in range(num_alternatives + 1):
    av[i] = 1

### Fitting

In [ ]:
logger = blog.get_screen_logger(level=blog.DEBUG)

In [21]:
# nest_move = move, list(range(1, num_alternatives + 1))
# nest_stay = 1.0, [0]
# nests = nest_move, nest_stay

In [22]:
# nest_logprob = models.lognested(V, av, nests, CHOSEN)

# biogeme_nest = bio.BIOGEME(database, nest_logprob, suggestScales=False)
# biogeme_nest.modelName = "nested_full_full"

# biogeme_nest.calculateNullLoglikelihood(av)

# results_nest = biogeme_nest.estimate()
# pandasResults_nest = results_nest.getEstimatedParameters()
# print(pandasResults_nest)

In [23]:
# Definition of the model. This is the contribution of each
# observation to the log likelihood function.
# estimating the CHOSEN field
logprob = models.loglogit(V, av, Variable("ALT_CHOICE"))

# formulas = {"loglike": logprob, "weight": W0}

# Create the Biogeme object
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = f"us_mnl_{year}"

Biogeme parameters read from biogeme.toml. 


In [24]:
# Calculate the null log likelihood for reporting. (likelihood of predicting every entry's alterantive correctly if alternatives are randomly chosen)
biogeme.calculate_null_loglikelihood(av)

-1965.9128163621629

In [25]:
# estimate parameters
results = biogeme.estimate()

# Get the results in a pandas table
pandasResults = results.get_estimated_parameters()
print(pandasResults)

*** Initial values of the parameters are obtained from the file __us_mnl_2018.iter 
Cannot read file __us_mnl_2018.iter. Statement is ignored. 
Starting values for the algorithm: {} 
As the model is not too complex, we activate the calculation of second derivatives. To change this behavior, modify the algorithm to "simple_bounds" in the TOML file. 
Optimization algorithm: hybrid Newton/BFGS with simple bounds [simple_bounds] 
** Optimization: Newton with trust region for simple bounds 
Iter.     Function    Relgrad   Radius      Rho      
    0      7.1e+03    1.5e+05      0.5    -0.16    - 
    1      7.1e+03    1.5e+05     0.25    -0.15    - 
    2      7.1e+03    1.5e+05     0.12    -0.15    - 
    3      7.1e+03    1.5e+05    0.062    -0.15    - 
    4      7.1e+03    1.5e+05    0.031    -0.15    - 
    5      7.1e+03    1.5e+05    0.016    -0.14    - 
    6      7.1e+03    1.5e+05   0.0078    -0.13    - 
    7      7.1e+03    1.5e+05   0.0039    -0.12    - 
    8      7.1e+03    1

                                         Name      Value  Robust std err.  \
0                                      c_stay  17.732825         1.434526   
1                            c_stay_age_18_22  -0.303127         1.851942   
2                            c_stay_age_23_29   0.141120         2.048595   
3                            c_stay_age_30_39   1.260269         1.883228   
4                            c_stay_age_40_49   2.867993         1.892248   
..                                        ...        ...              ...   
92         c_destchoice_naics_proportion_govt -17.028660         0.456520   
93  c_destchoice_naics_proportion_goods_trade  -1.108598         3.194344   
94      c_destchoice_proportion_naics_license -32.839350         0.425473   
95      c_destchoice_proportion_naics_high_ed   7.521474         3.848386   
96      c_destchoice_proportion_naics_agr_ext   7.991010         0.219408   

    Robust t-stat.  Robust p-value  
0        12.361452        0.000000  
1